In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import scipy
import pandas as pd

In [ ]:
#Input: saved WaveMAP dataframe
umap_df = pd.read_csv()

In [ ]:
#Sort your umap_df by cluster_id for plotting
umap_df = umap_df.sort_values(['cluster_id'])
#Make into a cute little numpy array and save as result
numpy_list = [array for array in umap_df['waveform']]
result = np.vstack(numpy_list)

In [ ]:
fig, ax = plt.subplots()

Ks=4 #What number of clusters are you plotting?
labels=umap_df['cluster_id']
cmap = sns.color_palette('hls', 4) #Change here the number of colors needed (# clusters)

# -> Cosmetic code to create a Rectangle patches to label specific K-cluster
Koffset = 0

sns.heatmap(result, cbar=True, cbar_kws={"label": r"z-scored $\frac{dF}{F}$"},cmap='mako',ax=ax)
for Ki in range(Ks):
    Nk = np.size(np.where(labels == Ki))
    # 40 is width of the rectangle
    rect = mpl.patches.Rectangle((0, Koffset), 100, Nk, linewidth=1, edgecolor='none', facecolor=cmap[Ki])
    ax.text(10, Koffset + Nk/2, Ki ,color='k', weight='bold')
    # Add the patch to the plot
    ax.add_patch(rect)
    Koffset += Nk
ax.text(10,-5,'↓ Cluster ID',fontsize=10)
# <- end of cosmetic code
ax.set_xlabel('Time (seconds)')
ax.set_ylabel('Cell #')
# ax.set_xticks(np.linspace(0, result.shape[1], 12),
# labels=np.linspace(0, 330, 12, dtype=np.int))
# ax.set_yticks(np.linspace(0, result.shape[0], 10),
# labels=np.linspace(0, 250, 10, dtype=np.int))

In [ ]:
#Do you want to look at a specific cluster? Look no further...
cluster1 = umap_df[umap_df['cluster_id']== 1]
numpy_list = [array for array in cluster1['waveform']]
cluster1 = np.vstack(numpy_list)
cluster1

In [ ]:
#Save that cluster numpy array for future use
np.save('/Users/suthardr/Desktop/cluster1_cxta_fc_shock.npy', cluster1)

In [ ]:
# Want to plot a heatmap for a specific cluster?

# Major ticks every 20, minor ticks every 5
fig, ax = plt.subplots()
sns.heatmap(cluster1, cbar=True, cbar_kws={"label": r"$z-scored \frac{dF}{F}$"}, cmap='mako', ax=ax)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Cell #')
# ax.set_xticks(np.linspace(0, cluster0.shape[1], 12),
# labels=np.linspace(0, 330, 12, dtype=np.int))
# ax.set_yticks(np.linspace(0, cluster0.shape[0], 10),
# labels=np.linspace(0, 32, 10, dtype=np.int))

#Save that heatmap as a png because svg will blow up your laptop
#fig.savefig('/Users/suthardr/Desktop/recall_cxtb_heatmap_clusterzoom.png')


In [ ]:
#Want to see what your average cluster plot looks like and find events?
avg = np.average(cluster1, axis=0)
peaks, properties = scipy.signal.find_peaks(avg, height=np.std(avg), distance=20, width=15, rel_height=0.5) #height =1 std, distance = 10Hz signal is 100ms/index, so 20 is 2 seconds,
    #width = 140ms decay time for GCaMP so 1.4, rel_height = 0.5 or FWHM
prominences = scipy.signal.peak_prominences(avg, peaks)[0]
height = avg[peaks]-prominences
properties #show the dictionary with the peak properties

    #
plt.figure()
plt.plot(avg)
plt.plot(peaks, avg[peaks], 'x')
plt.ylabel('dF/F')
plt.xlabel('Time (ms)')
plt.vlines(x=peaks, ymin=height, ymax=avg[peaks], color='orange')